# pandas Recap — Lessons 4 through 11
*Intro to Python for Scientists & Public Health Professionals*

This notebook recaps the core pandas sequence, from first Series and DataFrames through the full data-analysis workflow. Each section covers one lesson: the ideas introduced and the functionality — the methods, arguments, and idioms — that went with them. The running examples come from a diabetes hospital-encounters dataset, a year of daily clinic-visit counts, and county-level Census and CDC data, and the sequence builds toward a Rural Hospital Closures capstone.

The code blocks are compact cheat-sheets meant for reference, not a runnable pipeline; the datasets and full walkthroughs live in the individual lesson notebooks.

### Learning objectives
- Recall what each lesson (4–11) covered and how the sequence fits together
- Locate the right pandas tool for a task: loading, selecting, cleaning, grouping, dates, combining, reshaping
- Review the key methods and arguments introduced in each lesson at a glance

### Agenda
1. Series & DataFrames
2. Ingesting data
3. Indexes — `loc` & `iloc`
4. Inspecting & cleaning
5. Feature engineering & GroupBy
6. Dates & times
7. Combining data — merge, join, concat
8. Reshaping — melt, pivot, stack

## 1. Series & DataFrames (Lesson 4)

The foundation. A **Series** is a one-dimensional labeled array; a **DataFrame** is a set of Series sharing one index — a labeled table. Series and DataFrames are built from lists and dicts, and the defining pandas behavior appears here: **index alignment**, where arithmetic matches on the index label rather than on position, producing `NaN` wherever a label exists on only one side. Data is accessed by label (`.loc`) or by position (`.iloc`), a single column comes back as a Series while a list of names returns a DataFrame, and rows are ordered with the sort family.

> **Note:** Index alignment is the behavior that most surprises newcomers from NumPy — the same operation that looks positional is actually label-matched.

```python
import numpy as np
import pandas as pd

# Build
s   = pd.Series([218, 201, 216], index=["Mon", "Tue", "Wed"])   # labeled 1-D
pop = pd.Series({"Riverside": 42000, "Hilltop": 18500})         # dict -> Series
df  = pd.DataFrame({"site": [...], "visits": [...], "wait": [...]})  # dict of columns

# Index alignment
a + b                      # matched by label; unmatched labels -> NaN

# Inspect
df.shape; df.columns.tolist(); df.dtypes; df.info(); df.head()

# Select
df["age"]                  # single bracket  -> Series
df[["age", "gender"]]      # list of names   -> DataFrame
df.loc[0]                  # row by label
df.iloc[0]                 # row by position

# Sort
df.sort_values("visits", ascending=False)
df.sort_index()
df.nlargest(2, "visits"); df.nsmallest(2, "visits")
```

## 2. Ingesting Data (Lesson 5)

Getting messy real files *in*. `read_csv` is the workhorse, with arguments for the realities of raw data: a missing header row, unwanted columns, forced types, and dates to parse. The lesson also reads JSON, discusses why `read_excel` needs a real file (and why the course keeps its data as CSV/JSON in a public repo), and pulls live data from a public-health REST API. It closes with renaming columns three ways and writing results back out.

> **Tip:** Force identifier columns to `string` at read time with `dtype=` so leading zeros and formatting survive instead of being parsed as numbers.

```python
BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"

# read_csv with full control
pd.read_csv(url)                                   # header row assumed
pd.read_csv(url, header=None, names=cols)          # no header -> supply names
pd.read_csv(url, usecols=[...], index_col="id",
            parse_dates=["date"], nrows=1000,
            dtype={"patient_nbr": "string"})

# Other sources
pd.read_json(url)
# pd.read_excel("file.xlsx", sheet_name="Sheet1")  # needs a real file + openpyxl
import requests
df = pd.DataFrame(requests.get(api_url, timeout=30).json())   # REST -> DataFrame

# Rename columns (three idioms)
df = df.rename(columns={"old": "new"})
df.columns = ["a", "b", "c"]
df.columns = df.columns.str.upper()

# Write out
df.to_csv("clean.csv", index=False)
```

## 3. Indexes — `loc` & `iloc` (Lesson 6)

Selecting exactly the rows and columns wanted — the most-used skill in pandas. `loc` works by **label**, `iloc` by **integer position**, and both take `[rows, columns]`. The lesson drills the traps: `loc` slices are **inclusive** of both endpoints while `iloc` follows normal Python stop-excluded rules, a slice takes a third `step` element, boolean filtering keeps the rows a True/False Series marks, multiple conditions join with `&` / `|` (each parenthesized, never `and` / `or`), and `.isin()` matches a set of allowed values. The two accessors diverge the moment an integer index stops equaling positions.

> **Important:** `and` / `or` raise on a Series because they expect one True/False; use `&` and `|`, and wrap each condition in parentheses.

```python
# By label vs. by position
df.loc["Fri", "visits"]        # label
df.iloc[4, 1]                  # position
df.loc[:, "visits"]            # all rows, one column
df.loc[["Thu", "Fri"], ["visits", "wait_min"]]

# Slices differ at the endpoint
df.loc["Mon":"Thu"]           # INCLUSIVE of Thu
df.iloc[0:4]                  # stops BEFORE position 4
df.loc["Mon":"Fri":2]         # start:stop:step

# Boolean filtering
df.loc[df["visits"] > 150]
df.loc[(df["visits"] > 150) & (df["status"] == "Open"), ["visits", "wait_min"]]
df.loc[df["status"].isin(["Open", "Reduced"])]
```

## 4. Inspecting & Cleaning Data (Lesson 7)

The big cleaning workflow: raw load to analysis-ready. It moves through a first look (`head`, `info`, `describe`), duplicate removal, dtype fixes (IDs are labels, not quantities), quantifying missingness, and the crucial catch of **sentinel values** — codes like `?` that mean missing but aren't counted until converted to real `NA`. Then imputing (median fill, plus forward/back fill), standardizing categorical values with the `.str` accessor and `replace` / `map`, spotting outliers with the **1.5 × IQR** rule, a quick `corr()` scan, and dropping columns and rows. It launches Part 1 of the capstone.

> **Warning:** Sentinel values hide in plain sight — `isna()` reports 0% missing for a column full of `?` until `replace("?", pd.NA)` runs. Always scan for them before trusting a missingness summary.

> **Tip:** When standardizing text, map genuinely unknown values to missing rather than to a guessed category — do not fabricate data while cleaning it.

```python
# First look
df.head(); df.info(); df.describe()

# Duplicates & types
df.duplicated().sum(); df = df.drop_duplicates()
df[id_cols] = df[id_cols].astype("string")
df = df.rename(columns={"admission_type_id": "admit_type"})

# Missing data + sentinels
(df.isna().mean() * 100).round(1).sort_values(ascending=False)
(df == "?").sum()
df = df.replace("?", pd.NA)
df["num_medications"] = df["num_medications"].fillna(df["num_medications"].median())
# df.ffill(); df.bfill()

# Standardize categories
df["gender"] = df["gender"].str.strip().str.lower()
df["gender"] = df["gender"].replace({"m": "male", "f": "female", "unknown/invalid": pd.NA})
df["diag_1"].str.startswith("250", na=False).sum()   # .str toolkit

# Outliers (1.5 x IQR) and correlations
q1, q3 = df["num_medications"].quantile([0.25, 0.75]); iqr = q3 - q1
df = df[df["num_medications"].between(q1 - 1.5*iqr, q3 + 1.5*iqr)]
df.corr(numeric_only=True).round(2)

# Drop
df = df.drop(columns=["weight", "payer_code"])
df = df[df["age"] != "xyz"]
```

## 5. Feature Engineering & GroupBy (Lesson 8)

Two everyday skills: building new columns and summarizing by group. New features come from vectorized math, `.map` for recoding a category to numbers, `apply(axis=1)` for row-wise logic (with the faster vectorized `np.where` as the preferred alternative), and binning with `cut` (fixed edges) or `qcut` (equal-sized quantiles). Grouping follows the **split-apply-combine** model: `groupby` with an aggregation, `.agg` with named outputs, grouping by multiple keys, `transform` to attach a group statistic back onto every row, and `filter` to keep or drop whole groups. Capstone Part 2.

> **Note:** `transform` returns a value for every row aligned to the original frame, whereas `agg` collapses each group to one row — that alignment is what lets a group mean be pasted back beside each record.

```python
# New columns
df["total_procedures"] = df["num_procedures"] + df["num_lab_procedures"]   # vectorized
df["gender_code"] = df["gender"].map({"male": 0, "female": 1})             # recode
df["stay_flag"]   = np.where(df["time_in_hospital"] > 7, "long", "short")  # vectorized if/else
df["stay_band"]   = pd.cut(df["time_in_hospital"], bins=[0, 3, 7, 14],
                           labels=["short", "medium", "long"])             # cut / qcut

# Split-apply-combine
df.groupby("gender")["num_medications"].mean()
df.groupby("gender").size()
df.groupby("gender").agg(avg_meds=("num_medications", "mean"),
                         n=("encounter_id", "size"))            # named aggregations
df.groupby(["gender", "age"], as_index=False)["time_in_hospital"].mean()

# transform (per-row group stat) and filter (whole groups)
df["gender_avg_stay"] = df.groupby("gender")["time_in_hospital"].transform("mean")
df.groupby("age").filter(lambda g: len(g) >= 5000)
```

## 6. Working with Dates & Times (Lesson 9)

Time is central to public-health data — cases per week, visits per month. The lesson parses strings to real datetimes with `pd.to_datetime` (custom `format`, `dayfirst`, and `errors="coerce"` to turn bad values into `NaT`), extracts parts with the `.dt` accessor, and puts the date on the index to unlock date-string slicing. Its most powerful tool is **resampling** — `groupby` for time — bucketing rows into calendar periods (`"ME"`, `"W"`, `"D"`) and aggregating, including per-group resampling. It finishes with rolling averages, day-of-week patterns, and filling time-series gaps with `interpolate`.

> **Tip:** For an ordered series, `interpolate` estimates a gap from the values on either side — usually more sensible for time data than a flat fill.

```python
# Parse
pd.to_datetime("10/03/2025", dayfirst=True)
pd.to_datetime(series, format="%Y-%d-%m")
pd.to_datetime(vals, errors="coerce")          # bad values -> NaT
visits = pd.read_csv(url, parse_dates=["date"])

# .dt accessor
visits["date"].dt.year
visits["date"].dt.month
visits["date"].dt.day_name()
visits["date"].dt.isocalendar().week

# Datetime index -> slice + resample
ts = visits.set_index("date").sort_index()
ts.loc["2025-03"]                              # a month
ts.loc["2025-03":"2025-06"]                    # inclusive range
ts["visits"].resample("ME").sum()              # month-end totals
ts.groupby("site").resample("ME")["visits"].sum()

# Rolling window + gaps
ts["visits"].rolling(7).mean()
ts["visits"].interpolate()                     # also ffill() / bfill()
```

## 7. Combining Data — Merge, Join, Concat (Lesson 10)

Real analyses pull from more than one table. Three tools cover almost everything: **`concat`** stacks like-shaped tables, **`merge`** does SQL-style joins on key columns, and **`join`** combines on the index. Merge defaults to an inner join, and `how=` selects inner / left / right / outer, with unmatched rows filled by `NaN`; `left_on` / `right_on` handle differently-named keys. The worked example merges ACS demographics with CDC PLACES health estimates on a FIPS code, with the standard warning attached. Capstone Part 3.

> **Warning:** The #1 merge gotcha is a dtype mismatch on the join key — a text `"13001"` will silently match nothing against an integer `13001`. Confirm both keys share a dtype before trusting the result.

```python
# Stack
pd.concat([oct_df, nov_df], ignore_index=True)

# Merge on a shared key; how = inner | left | right | outer
demographics.merge(health, on="county", how="inner")   # default: keys in both
demographics.merge(health, on="county", how="left")    # all left rows; NaN fills
demographics.merge(health, on="county", how="outer")   # keys in either

# Differently-named keys
a.merge(b, left_on="county", right_on="cty")
acs.merge(places, left_on="CountyId", right_on="CountyFIPS", how="inner")

# Join on the index
left.join(right, how="inner")
```

## 8. Reshaping — Melt, Pivot, Stack (Lesson 11)

The same data can be **wide** (a column per measure) or **long/tidy** (one row per observation); most grouping and plotting wants long, while reports often want wide. **`melt`** goes wide to long, **`pivot`** / **`pivot_table`** go long to wide (the table version aggregates duplicates), and **`stack`** / **`unstack`** move data between columns and the index. The closing example reshapes CDC PLACES county health measures and shows why tidy data makes grouping straightforward. This is the last lesson in the core sequence.

> **Note:** `pivot` errors when an index/column pair repeats; `pivot_table` handles duplicates by aggregating them — that is the practical difference between the two.

```python
# Wide -> long
long = wide.melt(id_vars="site", var_name="month", value_name="visits")
long.groupby("month")["visits"].sum()          # trivial once tidy

# Long -> wide
long.pivot(index="site", columns="month", values="visits")
long.pivot_table(index="site", columns="month", values="visits", aggfunc="mean")

# Index-based cousins
wide.set_index("site").stack()                 # columns -> index (long Series)
stacked.unstack()                              # index -> columns (wide)
```

## Wrap-up

Across the eight lessons the sequence moves from first Series and DataFrames to a complete data-analysis workflow: load, select, clean, engineer, group, work with time, combine, and reshape. Three through-lines tie it together. The whole arc is **vectorization-first**, repeatedly steering away from row-wise `apply` toward vectorized alternatives. It consistently flags **modern-pandas idioms** — assign the result back rather than using `inplace`, and use `concat` in place of the removed `append`. And **visualization is kept minimal**: charts appear only to guide cleaning decisions, with polished visuals deferred to a separate Data Visualization course.